In [2]:
# 0) 保证能 import 到包
import sys
root = "~/CRAFT/VirtualCell/ours_v1/drug"  # 修改为你的项目根目录（其下有 MoleculeSTM/）
if root not in sys.path:
    sys.path.insert(0, root)

# 1) 导入
import torch
from pathlib import Path
from tokenizer import MolEncTokenizer
from util import REGEX, DEFAULT_CHEM_TOKEN_START, DEFAULT_VOCAB_PATH
from decoder import DecodeSampler
from megatron_bart import MegatronBART

# 2) 准备
ckpt_path = "/mnt/petrelfs/fengjinghao/molecule_model.pth"
vocab_path = "/mnt/petrelfs/fengjinghao/CRAFT/VirtualCell/ours_v1/drug/MoleculeSTM/MoleculeSTM/bart_vocab.txt"  # 确保指向实际 vocab 文件
device = "cuda" if torch.cuda.is_available() else "cpu"

# 3) 加载 state_dict
state_dict = torch.load(ckpt_path, map_location="cpu")

# 4) tokenizer 和超参
tokenizer = MolEncTokenizer.from_vocab_file(vocab_path, REGEX, DEFAULT_CHEM_TOKEN_START)
vocab_size = len(tokenizer)
pad_token_idx = tokenizer.vocab[tokenizer.pad_token]

d_model = state_dict["emb.weight"].shape[1]                        # 256
num_layers = max(int(k.split(".")[2]) for k in state_dict if k.startswith("encoder.layers.")) + 1  # 4
d_feedforward = state_dict["encoder.layers.0.fc1.weight"].shape[0] # 1024
num_heads = 8                                                      # 由权重形状确定
max_seq_len = state_dict["pos_emb"].shape[0]                       # 512

sampler = DecodeSampler(tokenizer, max_seq_len=max_seq_len)
model = MegatronBART(
    decode_sampler=sampler,
    pad_token_idx=pad_token_idx,
    vocab_size=vocab_size,
    d_model=d_model,
    num_layers=num_layers,
    num_heads=num_heads,
    d_feedforward=d_feedforward,
    max_seq_len=max_seq_len,
    dropout=0.1,
).to(device)

missing, unexpected = model.load_state_dict(state_dict, strict=True)
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

model.eval()
print("ready:", device)

# 5) 最小前向测试
B, Tenc, Tdec = 2, 8, 6
enc_ids = torch.randint(0, vocab_size, (Tenc, B), device=device)
dec_ids = torch.randint(0, vocab_size, (Tdec, B), device=device)
enc_mask = torch.zeros(Tenc, B, dtype=torch.bool, device=device)
dec_mask = torch.zeros(Tdec, B, dtype=torch.bool, device=device)

with torch.no_grad():
    out = model({
        "encoder_input": enc_ids,
        "encoder_pad_mask": enc_mask,
        "decoder_input": dec_ids,
        "decoder_pad_mask": dec_mask,
    })
    print("token_output:", out["token_output"].shape)  # (Tdec, B, vocab_size)

/tmp/ipykernel_259123/815281113.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(ckpt_path, map_location="cpu")


Missing keys: []
Unexpected keys: []
ready: cpu
token_output: torch.Size([6, 2, 523])


In [3]:
from rdkit import Chem

In [ ]:
# 示例：将单个 SMILES 编码为 encoder memory，并导出分子级向量与一次重构

import torch
from functools import partial

# 1) 准备一个 SMILES
SMILES1 = "C1=CC2=C(C(=C1)O)N=CC=C2"
SMILES2 = "C1=CC2=C(C(=C1)O)N=CC=C2C1=CC2=C(C(=C1)O)N=CC=C2"  # 可替换为你的分子

mol = Chem.MolFromSmiles(SMILES1)
canon_SMILES = Chem.MolToSmiles(mol)
canon_SMILES = Chem.MolToSmiles(Chem.MolFromSmiles(SMILES1))
smiles1 = canon_SMILES

# 2) 分词并构造 encoder 输入 (seq_len, batch=1)
tok = tokenizer.tokenize([smiles1, SMILES2], pad=True)
print(tok)

ids = tokenizer.convert_tokens_to_ids(tok['original_tokens'])  # List[List[int]]
print(ids)

encoder_input = torch.tensor(ids, dtype=torch.long, device=device).T                 # (seq_len, 1)
encoder_pad_mask = torch.tensor(tok['masked_pad_masks'], dtype=torch.bool, device=device).T  # (seq_len, 1)

# 截断到模型最大长度
encoder_input = encoder_input[:max_seq_len]
encoder_pad_mask = encoder_pad_mask[:max_seq_len]

# 3) 编码得到 memory (seq_len, 1, d_model)
with torch.no_grad():
    memory = model.encode({
        "encoder_input": encoder_input,
        "encoder_pad_mask": encoder_pad_mask,
    })

print("SMILES:", smiles1)
print("memory shape:", tuple(memory.shape))  # (L, 1, d_model) 例如 (L, 1, 256)

# 4) 从序列表示得到单个分子向量：对非 pad 位置做平均池化
valid = (~encoder_pad_mask).float()                       # (L, 1)
weights = valid / (valid.sum(dim=0, keepdim=True) + 1e-9)
mol_vec = (memory.squeeze(1) * weights).sum(dim=0)        # (d_model,)
print("molecule embedding (pooled) shape:", tuple(mol_vec.shape))  # (256,)

# 5) 基于该 memory 做一次 greedy 重构（检验编码-解码链路）
with torch.no_grad():
    decode_fn = partial(model._decode_fn, memory=memory, mem_pad_mask=encoder_pad_mask)
    mol_strs, log_lhs = model.sampler.beam_decode(decode_fn, batch_size=1, device=device)

decoded = mol_strs[0]  # List[str]，单样本
print("decoded SMILES (greedy):", decoded)

{'original_pad_masks': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'masked_pad_masks': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'original_tokens': [['^', 'O', 'c', '1', 'c', 'c', 'c', 'c', '2', 'c', 'c', 'c', 'n', 'c', '1', '2', '&', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD

RuntimeError: The size of tensor a (256) must match the size of tensor b (2) at non-singleton dimension 2

c1c(C(=O)NCCCOCC)nccn1

In [5]:
import ast

def extract_concentration(conc_str):
    """从drug_conc字符串中提取浓度数值"""
    # 安全解析字符串为Python对象
    conc_list = ast.literal_eval(conc_str)
    
    # 提取第一个药物的浓度（索引1是浓度）
    assert len(conc_list) == 1 and len(conc_list[0]) == 3
    concentration = conc_list[0][1]  # 0.05
    return float(concentration)

# 测试
conc_str = "[('8-Hydroxyquinoline', 0.05, 'uM')]"
concentration = extract_concentration(conc_str)
print(concentration)  # 输出: 0.05

0.05
